# TRGB Comparative Analysis — Reproduction Notebook

Self-contained walkthrough of the H-ΛCDM TRGB comparative pipeline. Designed to be runnable end-to-end by an external observational cosmology reviewer using only public archival data.

The notebook covers:

1. **Freedman 2019/2020 (Case A)** reproduction — LMC-anchored HST photometry.
2. **Freedman 2024/2025 (Case B)** reproduction — NGC 4258-anchored JWST photometry.
3. **H-ΛCDM framework forward predictions** — the holographic projection formula applied independently to each d_local.

The Freedman reproduction cells run without any H-ΛCDM framework knowledge; they stand as independent observational cosmology. The framework cells are clearly marked and operate on the same shared data via forward prediction only.

**Preregistration**: all methodological choices are frozen in `docs/trgb_comparative_preregistration_stage{1,2}.md`. This notebook does not change any of those choices.

## 0. Setup

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / 'main.py').exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

import numpy as np
import matplotlib.pyplot as plt

## 1. Data availability

In [ ]:
from data.loader import DataLoader
loader = DataLoader()
availability = loader.check_data_availability()
for k, v in sorted(availability.items()):
    marker = '✓' if v else '✗'
    print(f'  {marker} {k}')

If TRGB keys are `✗`, the archival data is not yet downloaded. Run `python main.py --trgb-comparative-load-data` from the project root to fetch, or place manifest files under `downloaded_data/trgb/` per the preregistration Stage 1 document.

## 2. Framework forward predictions

The framework's projection formula is a pure forward prediction given d_local. It does not need photometric data. We compute predictions for both d_local values (LMC = 0.0496 Mpc, NGC 4258 = 7.58 Mpc) with Monte Carlo propagation over the Planck 2018 H_CMB posterior.

In [ ]:
from pipeline.trgb_comparative.framework_methodology import FrameworkMethodology
fw = FrameworkMethodology()
pred_a = fw.predict(label='H0_framework_predicted_lmc_anchor',
                    d_local_mpc=0.0496, sigma_d_local_mpc=0.0009,
                    n_samples=20_000, seed=0)
pred_b = fw.predict(label='H0_framework_predicted_ngc4258_anchor',
                    d_local_mpc=7.58, sigma_d_local_mpc=0.08,
                    n_samples=20_000, seed=1)
print(f'Case A (LMC anchor):      H₀ = {pred_a.H0_median:.3f} km/s/Mpc '
      f'[{pred_a.H0_low:.3f}, {pred_a.H0_high:.3f}]  breakdown={pred_a.breakdown_fraction:.2f}')
print(f'Case B (NGC 4258 anchor): H₀ = {pred_b.H0_median:.3f} km/s/Mpc '
      f'[{pred_b.H0_low:.3f}, {pred_b.H0_high:.3f}]  breakdown={pred_b.breakdown_fraction:.2f}')
if pred_a.breakdown_messages:
    print('\nCase A perturbative breakdown note:')
    print(' ', pred_a.breakdown_messages[0])

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(pred_a.H0_samples, bins=60, alpha=0.6, label='Case A (LMC)', color='#F18F01')
ax.hist(pred_b.H0_samples, bins=60, alpha=0.6, label='Case B (NGC 4258)', color='#2E86AB')
ax.axvline(73.04, ls='--', color='#8B5CF6', label='SH0ES observed (73.04)')
ax.axvline(69.8, ls='--', color='#A23B72', label='Freedman 2020 (69.8)')
ax.axvline(70.39, ls=':', color='#6B7280', label='Freedman 2024 (70.39)')
ax.set_xlabel(r'$H_0$ [km/s/Mpc]'); ax.set_ylabel('MC draws')
ax.legend(fontsize=8)
ax.set_title('Framework forward predictions for both d_local cases')
plt.tight_layout()

## 3. Freedman reproduction — requires photometric data

The next cells assume archival HST and JWST photometry is locally cached. With strict_data=False, cells will execute and report missing data without running MCMC.

### 3.1. Load bundles

In [ ]:
from pipeline.trgb_comparative.data_loaders import load_case_a, load_case_b
try:
    bundle_a = load_case_a(loader, strict=False)
    print(f'Case A: anchor={bundle_a.anchor.name}, '
          f'anchor_fields={len(bundle_a.anchor_fields)}, hosts={len(bundle_a.host_fields)}')
except Exception as exc:
    print(f'Case A bundle: {exc}')
try:
    bundle_b = load_case_b(loader, strict=False)
    print(f'Case B: anchor={bundle_b.anchor.name}, '
          f'anchor_fields={len(bundle_b.anchor_fields)}, hosts={len(bundle_b.host_fields)}')
except Exception as exc:
    print(f'Case B bundle: {exc}')

### 3.2. Run the Case A reproduction

This cell runs emcee MCMC. With MCMCSettings.short() it takes about 1 minute; with the full preregistered settings (32 walkers × 10k steps) it takes several hours.

The Freedman 2019/2020 sample-selection and methodological choices are baked into `pipeline.trgb_comparative.freedman_2020_methodology` — verifiable in the source.

In [ ]:
from pipeline.trgb_comparative.mcmc_runner import MCMCSettings
from pipeline.trgb_comparative.freedman_2020_methodology import run_freedman_2020
case_a_result = None
if 'bundle_a' in globals() and bundle_a is not None and bundle_a.host_fields:
    case_a_result = run_freedman_2020(bundle_a, MCMCSettings.short(),
                                       log_fn=print, tolerance_mag=0.8)
    print(f'reproduced H₀ = {case_a_result.reproduced_H0:.3f} ± '
          f'{case_a_result.reproduced_sigma_stat:.3f} (stat)')
    print(f'published H₀ = {case_a_result.published_H0}; '
          f'Δ = {case_a_result.reproduction_delta:+.3f}; '
          f'within tolerance: {case_a_result.reproduction_within_tolerance}')
else:
    print('Case A host photometry not available — skipping.')

### 3.3. Run the Case B reproduction

In [ ]:
from pipeline.trgb_comparative.freedman_2024_methodology import run_freedman_2024
case_b_result = None
if 'bundle_b' in globals() and bundle_b is not None and bundle_b.host_fields:
    case_b_result = run_freedman_2024(bundle_b, MCMCSettings.short(),
                                       log_fn=print, tolerance_mag=1.22)
    print(f'reproduced H₀ = {case_b_result.reproduced_H0:.3f} ± '
          f'{case_b_result.reproduced_sigma_stat:.3f} (stat)')
    print(f'published H₀ = {case_b_result.published_H0}; '
          f'Δ = {case_b_result.reproduction_delta:+.3f}; '
          f'within tolerance: {case_b_result.reproduction_within_tolerance}')
else:
    print('Case B host photometry not available — skipping.')

## 4. Side-by-side comparison

In [ ]:
rows = []
if case_a_result is not None:
    rows.append(('Case A published', 69.8, 0.8))
    rows.append(('Case A reproduced', case_a_result.reproduced_H0, case_a_result.reproduced_sigma_stat))
rows.append(('Case A framework (LMC)', pred_a.H0_median, 0.5 * (pred_a.H0_high - pred_a.H0_low)))
if case_b_result is not None:
    rows.append(('Case B published', 70.39, 1.22))
    rows.append(('Case B reproduced', case_b_result.reproduced_H0, case_b_result.reproduced_sigma_stat))
rows.append(('Case B framework (NGC 4258)', pred_b.H0_median, 0.5 * (pred_b.H0_high - pred_b.H0_low)))
for label, v, s in rows:
    print(f'  {label:40s} H₀ = {v:7.3f} ± {s:.3f}')

## 5. Full pipeline run (optional)

To reproduce the complete results (identical to running `python main.py --trgb-comparative`), uncomment the next cell. This runs the emcee MCMC at the preregistered 32-walker × 10k-step settings; expect ≥ 8 hours wall time for each Freedman case.

In [ ]:
# from pipeline.trgb_comparative import TRGBComparativePipeline
# pipe = TRGBComparativePipeline('results/trgb_comparative')
# out = pipe.run({'short': False, 'enforce_preregistration': True, 'strict_data': True})